In [ ]:
! pip install geemap

import geemap
import ee
from geemap import basemaps

ee.Authenticate()
ee.Initialize(project='kotwica-test')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 14.8 MB/s eta 0:00:00


In [ ]:
import ee
import geemap
import folium
import ipywidgets

m = geemap.Map()


states = ee.FeatureCollection("TIGER/2018/Counties").filter(
    ee.Filter.eq('STATEFP', '24')
)

md = states.filter(
    ee.Filter.inList('NAME', ['Cecil', 'Dorchester', 'Kent', "Queen Anne's", 'Somerset', 'Talbot', 'Wicomico','Caroline','Worcester'])
)
cropped_image = image.clip(md)

m.add_marker(
    location=[38.97, -76.22],
    popup=ipywidgets.HTML("My Hometown"),
    tooltip="Click me"
)


m.setCenter(-75.92, 39.10, 8)
m.addLayer(md.style(color='black', fillColor='00000000', width=2), {}, 'Maryland Counties')

m

Map(center=[39.1, -75.92], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

In [ ]:
import ee
import geemap

preisabel = geemap.Map()

landsat = (
    ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
    .filterDate('2002-08-15', '2002-10-01')
    .filter(ee.Filter.lt('CLOUD_COVER', 50))
)

states = ee.FeatureCollection("TIGER/2018/Counties").filter(
    ee.Filter.eq('STATEFP', '24')
)

md = states.filter(
    ee.Filter.inList('NAME', ['Cecil', 'Dorchester', 'Kent', "Queen Anne's", 'Somerset', 'Talbot', 'Wicomico'])
)

image = (
    landsat.median()
    .select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    .multiply(0.0000275)
    .add(-0.2)
)
cropped_imagepre = image.clip(md)


vis_paramspre = {
    'bands': ['SR_B3', 'SR_B2', 'SR_B1'],
    'min': -0.1,
    'max': 0.4,
    'gamma': 1.4
}

preisabel.setCenter(-75.92, 39.10, 8)
preisabel.addLayer(cropped_imagepre, vis_paramspre, 'Landsat 7')
preisabel.addLayer(md.style(color='black', fillColor='00000000', width=2), {}, 'Maryland Counties')

preisabel

Map(center=[39.1, -75.92], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

In [ ]:
import ee
import geemap

postisabel = geemap.Map()

landsat = (
    ee.ImageCollection("LANDSAT/LT05/C02/T1_L2")
    .filterDate('2004-08-15', '2004-10-01')
    .filter(ee.Filter.lt('CLOUD_COVER', 50))
)

states = ee.FeatureCollection("TIGER/2018/Counties").filter(
    ee.Filter.eq('STATEFP', '24')
)

md = states.filter(
    ee.Filter.inList('NAME', ['Cecil', 'Dorchester', 'Kent', "Queen Anne's", 'Somerset', 'Talbot', 'Wicomico'])
)

image = (
    landsat.median()
    .select(['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B7'])
    .multiply(0.0000275)
    .add(-0.2)
)
cropped_imagepost = image.clip(md)


vis_paramspost = {
    'bands': ['SR_B3', 'SR_B2', 'SR_B1'],
    'min': -0.1,
    'max': 0.4,
    'gamma': 1.4
}

postisabel.setCenter(-75.92, 39.10, 8)
postisabel.addLayer(cropped_imagepost, vis_paramspost, 'Landsat 7')
postisabel.addLayer(md.style(color='black', fillColor='00000000', width=2), {}, 'Maryland Counties')

postisabel


Map(center=[39.1, -75.92], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

In [ ]:
def add_ndwi(image):
    swir = image.select('SR_B5')
    nir = image.select('SR_B4')
    ndwi = swir.subtract(nir).divide(swir.add(nir)).rename('NDWI')
    return image.addBands(ndwi)

ndwipre = add_ndwi(cropped_imagepre).select('NDWI')
ndwipost = add_ndwi(cropped_imagepost).select('NDWI')

ndbi_diff = ndwipost.subtract(ndwipre).rename('NDWI_Difference')

def add_ndwi(image):
    swir = image.select('SR_B5')
    nir = image.select('SR_B4')
    ndwi = swir.subtract(nir).divide(swir.add(nir)).rename('NDWI')
    return image.addBands(ndwi)

ndwipre = add_ndwi(cropped_imagepre).select('NDWI')
ndwipost = add_ndwi(cropped_imagepost).select('NDWI')
ndwi_diff = ndwipost.subtract(ndwipre).rename('NDWI_Difference')

vis_params_diff = {'min': -1, 'max': 1, 'palette': ['green', 'white', 'blue']}

map_diff = geemap.Map(center=[-76.08, 39.00], zoom=8)
map_diff.addLayer(ndwipre, {'min': -1, 'max': 1, 'palette': ['green','white','blue']}, 'NDWI pre')
map_diff.addLayer(ndwipost, {'min': -1, 'max': 1, 'palette': ['green','white','blue']}, 'NDWI post')
map_diff.addLayer(ndwi_diff, vis_params_diff, 'NDWI Difference (pre isabel - post isabel)')


map_diff.addLayer(md.style(color='black', fillColor='00000000', width=2), {}, 'Selected Counties')

map_diff.add_colorbar(vis_params_diff, label='NDWI Change (pre Isabel - post Isabel)')
map_diff

Map(center=[-76.08, 39.0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGU…

In [ ]:
import ee
import geemap


# Assuming ndwi_diff and md (counties) are already defined

# Step 1: Reduce NDWI difference over each county (mean)
county_stats = ndwi_diff.reduceRegions(
    collection=md,
    reducer=ee.Reducer.mean(),
    scale=30  # Landsat resolution
)

# Step 2: Get results as a Python list
county_list = county_stats.getInfo()['features']

# Step 3: Extract county names and mean NDWI difference
county_summary = []
for f in county_list:
    name = f['properties']['NAME']
    mean_diff = f['properties']['mean']
    # Determine if change is positive or negative
    change_direction = "Increase (More Water)" if mean_diff > 0 else "Decrease (Less Water)" if mean_diff < 0 else "No Change"
    county_summary.append({'County': name, 'NDWI_Change': mean_diff, 'Direction': change_direction})

# Step 4: Determine thresholds for classification (top/middle/bottom thirds)
changes = [c['NDWI_Change'] for c in county_summary]
changes_sorted = sorted(changes)

n = len(changes_sorted)
low_thresh = changes_sorted[n//3]
high_thresh = changes_sorted[2*n//3]

# Step 5: Categorize counties
for c in county_summary:
    if c['NDWI_Change'] <= low_thresh:
        c['Category'] = 'Least Change'
    elif c['NDWI_Change'] >= high_thresh:
        c['Category'] = 'Most Change'
    else:
        c['Category'] = 'No Change'

# Step 6: Print summary table
print("County NDWI Change Summary:")
print("{:<15} {:>12} {:>20} {:>15}".format("County", "NDWI_Change", "Direction", "Category"))
for c in county_summary:
    print("{:<15} {:>12.4f} {:>20} {:>15}".format(c['County'], c['NDWI_Change'], c['Direction'], c['Category']))





County NDWI Change Summary:
County           NDWI_Change            Direction        Category
Queen Anne's         -0.0014 Decrease (Less Water)       No Change
Dorchester            0.0754 Increase (More Water)     Most Change
Talbot                0.0540 Increase (More Water)     Most Change
Caroline              0.0312 Increase (More Water)       No Change
Kent                  0.0346 Increase (More Water)     Most Change
Worcester            -0.0496 Decrease (Less Water)    Least Change
Wicomico             -0.0518 Decrease (Less Water)    Least Change
Somerset             -0.0387 Decrease (Less Water)    Least Change
Cecil                -0.0161 Decrease (Less Water)    Least Change
